# IsoNet Normalization & Missing-Wedge Visualisation

Two interactive tools for assessing whether the per-volume normalization is
appropriate for the isonet training strategy.

| Section | What it shows |
|---------|---------------|
| 1 | Setup: FBP reconstruction + sphere crop |
| 2 | FBP viewer: slice viewer + all-K histogram overlay |
| 3 | Wedge-carved pair: apply k_wedge mask to volumes i and j, compare distributions |

**Key question**: are the per-direction distributions similar enough (in shape
and scale) that a model trained on one direction generalises to another after
normalization?

In [1]:
import sys
sys.path.insert(0, '/myhome/smartt')

import numpy as np
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

from mumott.data_handling import DataContainer
from smartt.saxs_fbp import fibonacci_hemisphere, saxs_fbp_reconstruction
from smartt.saxs_isonet.preprocess import spherical_crop, make_sphere_mask
from smartt.saxs_isonet.wedge import (
    missing_wedge_mask_3d,
    all_missing_arcs,
    goniometer_axis_for_half_space,
)

# ── Parameters ────────────────────────────────────────────────────────────────
DATA_PATH  = '/myhome/data/smartt/shared/frogbone/dataset_qbin_0009.h5'
K          = 30
HALF_SPACE = 'y'
ALPHA_DEG  = 45.0
# ─────────────────────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
goniometer_axis = goniometer_axis_for_half_space(HALF_SPACE)

INFO:Setting the number of threads to 8. If your physical cores are fewer than this number, you may want to use numba.set_num_threads(n), and os.environ["OPENBLAS_NUM_THREADS"] = f"{n}" to set the number of threads to the number of physical cores n.
INFO:Setting numba log level to WARNING.
device: cuda


## 1. FBP Reconstruction + Sphere Crop

In [ ]:
dc = DataContainer(DATA_PATH, nonfinite_replacement_value=0)
dc.geometry.full_circle_covered = False

y_directions = fibonacci_hemisphere(K, half_space=HALF_SPACE)

recon_fbp, _, _ = saxs_fbp_reconstruction(
    dc=dc,
    k_fibonacci=K,
    filter_type='hann',
    n_projection_samples=64,
    device=device,
    verbose=True,
    return_matrix=True,
    projection_method='ball',
    ball_threshold=0.5,
    half_space=HALF_SPACE,
)
recon_fbp_np = recon_fbp.cpu().numpy()   # (K, X, Y, Z)
print(f'FBP shape: {recon_fbp_np.shape}  range: [{recon_fbp_np.min():.3e}, {recon_fbp_np.max():.3e}]')

INFO:Rotation matrices were loaded from the input file.


/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:228: DeprecationWarning: Entry name rotations is deprecated. Use inner_angle instead.
  _deprecated_key_warning('rotations')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:237: DeprecationWarning: Entry name tilts is deprecated. Use outer_angle instead.
  _deprecated_key_warning('tilts')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:247: DeprecationWarning: Entry name rot_mat is deprecated. Use rotation_matrix instead.
  _deprecated_key_warning('rot_mat')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:269: DeprecationWarning: Entry name offset_j is deprecated. Use j_offset instead.
  _deprecated_key_warning('offset_j')
/opt/conda/lib/python3.10/site-packages/mumott/data_handling/data_container.py:279: DeprecationWarning: Entry name offset_k is deprecated. Use k_offset instead.
  _deprecated_key_warning('offset_k'

INFO:Sample geometry loaded from file.
INFO:Detector geometry loaded from file.
INFO:Building projection matrix on GPU: K=30, n_samples=64, method=ball
INFO:Moving data to cuda...


SAXS FBP: 100%|██████████| 30/30 [00:00<00:00, 64.17dir/s, n_sub=171]


FBP shape: (30, 65, 82, 65)  range: [-9.871e+01, 3.303e+02]


In [3]:
# Sphere crop to the largest inscribed cube that is a multiple of 8.
recon_fbp_t, _, SPHERE_D = spherical_crop(
    torch.from_numpy(recon_fbp_np), cube_size=None
)
recon_fbp_np = recon_fbp_t.numpy()           # (K, d, d, d)
sphere_mask_np = make_sphere_mask(SPHERE_D)  # (d, d, d) bool

arcs_deg = np.degrees(all_missing_arcs(y_directions, ALPHA_DEG, goniometer_axis))

print(f'Sphere diameter d={SPHERE_D};  FBP shape: {recon_fbp_np.shape}')
print(f'Missing-arc range: [{arcs_deg.min():.1f}°, {arcs_deg.max():.1f}°]')
print(f'Directions sorted by missing arc:')
for k in np.argsort(arcs_deg):
    print(f'  k={k:2d}  arc={arcs_deg[k]:5.1f}°  dir={np.round(y_directions[k], 2)}')

Sphere diameter d=64;  FBP shape: (30, 64, 64, 64)
Missing-arc range: [0.0°, 180.0°]
Directions sorted by missing arc:
  k=24  arc=  0.0°  dir=[0.29 0.82 0.5 ]
  k=25  arc=  0.0°  dir=[-0.5   0.85 -0.16]
  k=26  arc=  0.0°  dir=[ 0.43  0.88 -0.2 ]
  k=27  arc=  0.0°  dir=[-0.15  0.92  0.37]
  k=28  arc=  0.0°  dir=[-0.11  0.95 -0.29]
  k=21  arc=  0.0°  dir=[0.69 0.72 0.09]
  k=23  arc=  0.0°  dir=[ 0.14  0.78 -0.61]
  k=29  arc=  0.0°  dir=[0.16 0.98 0.08]
  k=22  arc=  0.0°  dir=[-0.54  0.75  0.38]
  k=20  arc= 57.7°  dir=[-0.47  0.68 -0.56]
  k=19  arc= 86.0°  dir=[-0.04  0.65  0.76]
  k=18  arc=104.3°  dir=[ 0.56  0.62 -0.56]
  k=17  arc=117.9°  dir=[-0.81  0.58  0.03]
  k=16  arc=128.6°  dir=[0.64 0.55 0.54]
  k=15  arc=137.3°  dir=[-0.11  0.52 -0.85]
  k=14  arc=144.5°  dir=[-0.5   0.48  0.72]
  k=13  arc=150.6°  dir=[ 0.87  0.45 -0.19]
  k=12  arc=155.7°  dir=[-0.79  0.42 -0.46]
  k=11  arc=160.2°  dir=[0.28 0.38 0.88]
  k=10  arc=164.0°  dir=[ 0.4   0.35 -0.85]
  k= 9  arc=167.

## 2. FBP Viewer — Slice View + All-K Histogram

Slide through k-directions and x/y/z planes.  The bottom panel overlays the
sphere-interior distributions for **all K volumes** as gray step-histograms;
the selected k is highlighted in blue.  Orange lines mark the FBP colour-scale
limits (2nd/98th percentile of the mean volume).

If the blue curve is far from the cluster of gray curves, that direction has an
unusual dynamic range relative to the training set.

In [4]:
from matplotlib.lines import Line2D

def _make_fbp_viewer(volume, sphere_mask, directions, arcs_deg):
    K_, X_, Y_, Z_ = volume.shape
    mean_vol = volume.mean(axis=0)
    _abs_lo  = float(np.percentile(mean_vol[sphere_mask], 2))
    _abs_hi  = float(np.percentile(mean_vol[sphere_mask], 98))

    # Pre-extract sphere interiors once for speed.
    interiors = [volume[k_][sphere_mask].ravel() for k_ in range(K_)]
    all_vals  = np.concatenate(interiors)
    bins      = np.linspace(np.percentile(all_vals, 0.5), np.percentile(all_vals, 99.5), 80)

    # Per-volume normalization — same formula as the pipeline (mean/std from sphere interior).
    norm_stats     = [(float(v.mean()), float(v.std() + 1e-8)) for v in interiors]
    interiors_norm = [(v - mu) / sigma for v, (mu, sigma) in zip(interiors, norm_stats)]
    all_norm       = np.concatenate(interiors_norm)
    bins_norm      = np.linspace(np.percentile(all_norm, 0.5), np.percentile(all_norm, 99.5), 80)

    _crosshair_handles = [Line2D([], [], color=c, lw=1.5)
                          for c in ['limegreen', 'tomato', 'deepskyblue']]
    _crosshair_labels  = ['x', 'y', 'z']

    def _view(k, x, y, z):
        data = volume[k]

        fig = plt.figure(figsize=(13, 12))
        gs  = fig.add_gridspec(3, 3, height_ratios=[1, 0.55, 0.55], hspace=0.55, wspace=0.3)
        ax_yz     = fig.add_subplot(gs[0, 0])
        ax_xz     = fig.add_subplot(gs[0, 1])
        ax_xy     = fig.add_subplot(gs[0, 2])
        ax_hist   = fig.add_subplot(gs[1, :])
        ax_hist_n = fig.add_subplot(gs[2, :])

        kw = dict(cmap='inferno', vmin=_abs_lo, vmax=_abs_hi, aspect='equal', origin='lower')

        im = ax_yz.imshow(data[x, :, :].T, **kw)
        ax_yz.axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        ax_yz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_yz.set_title(f'YZ  (x={x})')
        ax_yz.set_xlabel('Y')
        ax_yz.set_ylabel('Z')

        ax_xz.imshow(data[:, y, :].T, **kw)
        ax_xz.axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        ax_xz.axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        ax_xz.set_title(f'XZ  (y={y})')
        ax_xz.set_xlabel('X')
        ax_xz.set_ylabel('Z')

        ax_xy.imshow(data[:, :, z].T, **kw)
        ax_xy.axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        ax_xy.axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        ax_xy.set_title(f'XY  (z={z})')
        ax_xy.set_xlabel('X')
        ax_xy.set_ylabel('Y')

        ax_xy.legend(_crosshair_handles, _crosshair_labels,
                     fontsize=7, loc='lower right', framealpha=0.5, title='slice pos.')
        plt.colorbar(im, ax=[ax_yz, ax_xz, ax_xy], shrink=0.65, label='Intensity')

        # ── Raw histogram ────────────────────────────────────────────────────
        for k_ in range(K_):
            ax_hist.hist(
                interiors[k_], bins=bins, histtype='step',
                color='steelblue' if k_ == k else 'gray',
                lw=1.8 if k_ == k else 0.5,
                alpha=1.0 if k_ == k else 0.25,
                label=f'k={k_} (selected)' if k_ == k else '_nolegend_',
            )

        mu, sigma = norm_stats[k]
        ax_hist.axvline(mu,         color='steelblue', lw=1.5, ls='--', label=f'μ = {mu:.3g}')
        ax_hist.axvline(mu - sigma, color='steelblue', lw=1.0, ls=':',  label=f'σ = {sigma:.3g}')
        ax_hist.axvline(mu + sigma, color='steelblue', lw=1.0, ls=':')
        ax_hist.axvline(_abs_lo, color='orange', lw=1.2, ls='--', alpha=0.8, label='cmap lo / hi')
        ax_hist.axvline(_abs_hi, color='orange', lw=1.2, ls='--', alpha=0.8)
        ax_hist.set_xlabel('Voxel intensity — raw')
        ax_hist.set_ylabel('Count')
        ax_hist.set_title(
            f'Raw distributions — all K overlaid  '
            f'(k={k}:  μ={mu:.3g},  σ={sigma:.3g},  missing arc={arcs_deg[k]:.1f}°)'
        )
        ax_hist.legend(fontsize=8)

        # ── Normalized histogram: (vol_k − μ_k) / σ_k ───────────────────────
        for k_ in range(K_):
            ax_hist_n.hist(
                interiors_norm[k_], bins=bins_norm, histtype='step',
                color='steelblue' if k_ == k else 'gray',
                lw=1.8 if k_ == k else 0.5,
                alpha=1.0 if k_ == k else 0.25,
                label=f'k={k_} (selected)' if k_ == k else '_nolegend_',
            )

        ax_hist_n.axvline(0.0,  color='steelblue', lw=1.5, ls='--', label='μ = 0  (by construction)')
        ax_hist_n.axvline(-1.0, color='steelblue', lw=1.0, ls=':',  label='±1σ')
        ax_hist_n.axvline(+1.0, color='steelblue', lw=1.0, ls=':')
        ax_hist_n.set_xlabel('Normalized voxel intensity  (vol − μ_k) / σ_k')
        ax_hist_n.set_ylabel('Count')
        ax_hist_n.set_title(
            'Normalized distributions — all K overlaid  '
            '(good normalization → curves overlap; shape differences = model must generalize across them)'
        )
        ax_hist_n.legend(fontsize=8)

        plt.suptitle(
            f'FBP  |  k={k}  y_dir={np.round(directions[k], 2)}  '
            f'missing arc={arcs_deg[k]:.1f}°',
            fontsize=11,
        )
        plt.show()

    interact(
        _view,
        k=widgets.IntSlider(min=0, max=K_-1, step=1, value=0,
                            description='k (dir)', continuous_update=False),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


_make_fbp_viewer(recon_fbp_np, sphere_mask_np, y_directions, arcs_deg)


interactive(children=(IntSlider(value=0, continuous_update=False, description='k (dir)', max=29), IntSlider(va…

## 3. Missing-Wedge Carved Pair Viewer

Simulates the training data-augmentation step:

1. Pick two source volumes **i** and **j** (the volumes whose dynamic range we
   want to compare).
2. Pick a wedge template **k_wedge** (the RSM direction whose Fourier mask will
   be carved out — equivalent to choosing which direction's missing wedge the
   model is asked to fill).
3. Apply that missing-wedge mask to both volumes in Fourier space
   (`mask = True` → keep, `False` → zero).
4. Display: top row = carved volume i, middle row = carved volume j (same xyz
   sliders, same colour scale); bottom row = histogram with
   - **solid** lines: distribution before carving
   - **dotted** lines: distribution after carving

> Note: rotation is **not** applied here (training uses a random SO(3) rotation
> first, then carves the canonical wedge of k_wedge).  This viewer applies
> the *natural* wedge of k_wedge in the volume's native frame — the effect on
> the distribution is identical to the rotated canonical case.

In [5]:
def _apply_wedge(vol_np, rsm_dir, alpha_deg, goniometer_axis):
    """Zero unmeasured Fourier frequencies (mask=False) and return real-space result."""
    mask = missing_wedge_mask_3d(rsm_dir, alpha_deg, vol_np.shape, goniometer_axis)
    F = np.fft.fftn(vol_np)
    F[~mask] = 0.0
    return np.fft.ifftn(F).real


def _make_wedge_viewer(volume, sphere_mask, directions, arcs_deg, alpha_deg, goniometer_axis):
    K_, X_, Y_, Z_ = volume.shape
    mean_vol = volume.mean(axis=0)
    _abs_lo  = float(np.percentile(mean_vol[sphere_mask], 2))
    _abs_hi  = float(np.percentile(mean_vol[sphere_mask], 98))

    def _slice_row(axes, data, x, y, z, kw):
        im = axes[0].imshow(data[x, :, :].T, **kw)
        axes[0].axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        axes[0].axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        axes[0].set_xlabel('Y'); axes[0].set_ylabel('Z')

        axes[1].imshow(data[:, y, :].T, **kw)
        axes[1].axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        axes[1].axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        axes[1].set_xlabel('X'); axes[1].set_ylabel('Z')

        axes[2].imshow(data[:, :, z].T, **kw)
        axes[2].axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        axes[2].axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')
        return im

    def _view(i, j, k_wedge, show_normalized, x, y, z):
        vol_i = volume[i]
        vol_j = volume[j]
        vol_i_carved = _apply_wedge(vol_i, directions[k_wedge], alpha_deg, goniometer_axis)
        vol_j_carved = _apply_wedge(vol_j, directions[k_wedge], alpha_deg, goniometer_axis)

        # Per-volume stats (pipeline convention: from raw sphere interior).
        int_i_raw  = vol_i[sphere_mask].ravel()
        int_j_raw  = vol_j[sphere_mask].ravel()
        int_i_crvd = vol_i_carved[sphere_mask].ravel()
        int_j_crvd = vol_j_carved[sphere_mask].ravel()

        mu_i, sigma_i = float(int_i_raw.mean()), float(int_i_raw.std() + 1e-8)
        mu_j, sigma_j = float(int_j_raw.mean()), float(int_j_raw.std() + 1e-8)

        if show_normalized:
            disp_i = (vol_i_carved - mu_i) / sigma_i
            disp_j = (vol_j_carved - mu_j) / sigma_j
            # Shared colour scale from combined normalized sphere interiors.
            combined_n = np.concatenate([(int_i_crvd - mu_i) / sigma_i,
                                         (int_j_crvd - mu_j) / sigma_j])
            vlo = float(np.percentile(combined_n, 1))
            vhi = float(np.percentile(combined_n, 99))
            cbar_label = 'Normalized intensity  (vol − μ_k) / σ_k'
        else:
            disp_i = vol_i_carved
            disp_j = vol_j_carved
            vlo, vhi = _abs_lo, _abs_hi
            cbar_label = 'Intensity (after wedge carving)'

        fig = plt.figure(figsize=(13, 17))
        gs  = fig.add_gridspec(4, 3, height_ratios=[1, 1, 0.6, 0.6], hspace=0.55, wspace=0.3)
        axes_i    = [fig.add_subplot(gs[0, c]) for c in range(3)]
        axes_j    = [fig.add_subplot(gs[1, c]) for c in range(3)]
        ax_hist   = fig.add_subplot(gs[2, :])
        ax_hist_n = fig.add_subplot(gs[3, :])

        kw = dict(cmap='inferno', vmin=vlo, vmax=vhi, aspect='equal', origin='lower')

        im = _slice_row(axes_i, disp_i, x, y, z, kw)
        for ax, plane in zip(axes_i, [f'YZ (x={x})', f'XZ (y={y})', f'XY (z={z})']):
            ax.set_title(
                f'i={i}  arc={arcs_deg[i]:.1f}°  ← wedge k={k_wedge}  |  {plane}',
                fontsize=9,
            )

        _slice_row(axes_j, disp_j, x, y, z, kw)
        for ax, plane in zip(axes_j, [f'YZ (x={x})', f'XZ (y={y})', f'XY (z={z})']):
            ax.set_title(
                f'j={j}  arc={arcs_deg[j]:.1f}°  ← wedge k={k_wedge}  |  {plane}',
                fontsize=9,
            )

        plt.colorbar(im, ax=axes_i + axes_j, shrink=0.5, label=cbar_label)

        # ── Raw histogram ────────────────────────────────────────────────────
        all_raw = np.concatenate([int_i_raw, int_j_raw])
        bins = np.linspace(np.percentile(all_raw, 0.5), np.percentile(all_raw, 99.5), 80)

        ax_hist.hist(int_i_raw,  bins=bins, histtype='step', color='steelblue', lw=2.0, ls='-',
                     label=f'i={i} before  (μ={mu_i:.3g}, σ={sigma_i:.3g})')
        ax_hist.hist(int_i_crvd, bins=bins, histtype='step', color='steelblue', lw=1.5, ls=':',
                     label=f'i={i} after wedge  (μ={int_i_crvd.mean():.3g}, σ={int_i_crvd.std():.3g})')
        ax_hist.hist(int_j_raw,  bins=bins, histtype='step', color='coral',     lw=2.0, ls='-',
                     label=f'j={j} before  (μ={mu_j:.3g}, σ={sigma_j:.3g})')
        ax_hist.hist(int_j_crvd, bins=bins, histtype='step', color='coral',     lw=1.5, ls=':',
                     label=f'j={j} after wedge  (μ={int_j_crvd.mean():.3g}, σ={int_j_crvd.std():.3g})')

        ax_hist.axvline(_abs_lo, color='orange', lw=1.0, ls='--', alpha=0.7, label='cmap lo / hi')
        ax_hist.axvline(_abs_hi, color='orange', lw=1.0, ls='--', alpha=0.7)
        ax_hist.set_xlabel('Voxel intensity — raw')
        ax_hist.set_ylabel('Count')
        ax_hist.set_title(
            f'Raw distributions — solid = before carving, dotted = after  '
            f'(wedge k={k_wedge}, arc={arcs_deg[k_wedge]:.1f}°)'
        )
        ax_hist.legend(fontsize=8)

        # ── Normalized histogram: (vol − μ_k) / σ_k ─────────────────────────
        i_raw_n  = (int_i_raw  - mu_i) / sigma_i
        i_crvd_n = (int_i_crvd - mu_i) / sigma_i
        j_raw_n  = (int_j_raw  - mu_j) / sigma_j
        j_crvd_n = (int_j_crvd - mu_j) / sigma_j

        all_norm  = np.concatenate([i_raw_n, j_raw_n])
        bins_norm = np.linspace(np.percentile(all_norm, 0.5), np.percentile(all_norm, 99.5), 80)

        ax_hist_n.hist(i_raw_n,  bins=bins_norm, histtype='step', color='steelblue', lw=2.0, ls='-',
                       label=f'i={i} before')
        ax_hist_n.hist(i_crvd_n, bins=bins_norm, histtype='step', color='steelblue', lw=1.5, ls=':',
                       label=f'i={i} after wedge')
        ax_hist_n.hist(j_raw_n,  bins=bins_norm, histtype='step', color='coral',     lw=2.0, ls='-',
                       label=f'j={j} before')
        ax_hist_n.hist(j_crvd_n, bins=bins_norm, histtype='step', color='coral',     lw=1.5, ls=':',
                       label=f'j={j} after wedge')

        ax_hist_n.axvline(0.0,  color='gray', lw=1.2, ls='--', alpha=0.8, label='μ = 0')
        ax_hist_n.axvline(-1.0, color='gray', lw=1.0, ls=':',  alpha=0.8, label='±1σ')
        ax_hist_n.axvline(+1.0, color='gray', lw=1.0, ls=':',  alpha=0.8)
        ax_hist_n.set_xlabel('Normalized voxel intensity  (vol − μ_k) / σ_k')
        ax_hist_n.set_ylabel('Count')
        ax_hist_n.set_title(
            'Normalized distributions — solid = before, dotted = after  '
            '(overlap between blue/coral → distributions comparable in normalized space)'
        )
        ax_hist_n.legend(fontsize=8)

        norm_tag = '  [normalized view]' if show_normalized else ''
        plt.suptitle(
            f'Wedge template: k={k_wedge}  arc={arcs_deg[k_wedge]:.1f}°  '
            f'dir={np.round(directions[k_wedge], 2)}{norm_tag}\n'
            f'Applied to  i={i} (arc={arcs_deg[i]:.1f}°)  and  j={j} (arc={arcs_deg[j]:.1f}°)',
            fontsize=11,
        )
        plt.show()

    interact(
        _view,
        i=widgets.IntSlider(min=0, max=K_-1, step=1, value=0,
                            description='i (vol A)', continuous_update=False),
        j=widgets.IntSlider(min=0, max=K_-1, step=1, value=K_//2,
                            description='j (vol B)', continuous_update=False),
        k_wedge=widgets.IntSlider(min=0, max=K_-1, step=1, value=K_-1,
                                  description='k_wedge', continuous_update=False),
        show_normalized=widgets.Checkbox(value=False, description='Show normalized'),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


_make_wedge_viewer(
    recon_fbp_np, sphere_mask_np, y_directions, arcs_deg, ALPHA_DEG, goniometer_axis
)


interactive(children=(IntSlider(value=0, continuous_update=False, description='i (vol A)', max=29), IntSlider(…

## 4. Model Inference Viewer

Loads a trained checkpoint and runs the model on volumes **i** and **j** after
the same input preparation used in inference:

1. Normalize each volume by its sphere-interior `(mean, std)`.
2. Apply the natural missing-wedge of `k_wedge` in Fourier space (same as the
   carved pair viewer above).
3. Rotate the carved+normalized volume to the **canonical frame** using
   `canonical_rotation(rsm_dir, goniometer_axis)`, so the model sees the same
   wedge orientation it was trained on.
4. Single UNet3D forward pass (the volume is already `SPHERE_D³`, one patch).
5. Rotate output back to the natural frame and enforce **Fourier consistency**:
   measured frequencies are replaced with those from the carved input; only the
   missing-wedge region comes from the model.

**Adjust `CHECKPOINT` below** to the round you want to evaluate.

Interpretation guide:
- `model output ≈ carved input` → model predicts near-zero in the missing region (not learning).
- `model output` adds energy in the wedge → model is filling missing frequencies ✓
- Compare to the FBP viewer (Section 2) to judge whether the filled volume
  looks realistic.

In [6]:
import pathlib

# ── Checkpoint (adjust to the round you want to evaluate) ────────────────────
CHECKPOINT = pathlib.Path(
    '/myhome/data/smartt/shared/isonet_results/frogbone_corrected/round_10/checkpoint.pt'
)
# ─────────────────────────────────────────────────────────────────────────────

from smartt.saxs_isonet.train import build_model
from smartt.saxs_isonet.augment import rotate_batch
from smartt.saxs_isonet.wedge import (
    canonical_goniometer_axis,
    canonical_rotation,
    sinusoidal_wedge_embedding,
    missing_arc_length,
)

_ckpt = torch.load(CHECKPOINT, map_location=device)
_COND_DIM = int(_ckpt.get('conditioning_dim', 128))

_infer_model = build_model(cross_attention_dim=_COND_DIM).to(device)
_infer_model.load_state_dict(_ckpt['model'])
_infer_model.eval()
print(f'Loaded: {CHECKPOINT}')
print(f'  alpha={_ckpt.get("alpha_deg", "?")}°  conditioning_dim={_COND_DIM}  cube_size={_ckpt.get("cube_size", "?")}')


def _run_model_single(vol_carved_norm_np, rsm_dir, alpha_deg, goniometer_axis,
                      apply_consistency=True):
    """Rotate to canonical frame → carve canonical wedge exactly → forward pass → rotate back.

    Input must already have sphere exterior = 0.0 in normalized space
    (matching dataset._load_norm which re-zeros after (vol−mean)/std).
    Carving is applied AFTER rotation to match the training order:
      augmentor does rotate_batch(vol, R_rand) then carve_shifted(rotated, canonical_mask).
    Fourier consistency: keep measured frequencies from carved input,
    fill missing-wedge region from model output.
    """
    vol_t = torch.from_numpy(np.asarray(vol_carved_norm_np, dtype=np.float32)).to(device)

    R_can = (torch.from_numpy(canonical_rotation(rsm_dir, goniometer_axis))
             .float().to(device).unsqueeze(0))   # (1, 3, 3)
    # R_can = torch.diag(torch.ones(3)).unsqueeze(0).to(device)
    R_inv = R_can.transpose(1, 2)

    vol_can = rotate_batch(vol_t.unsqueeze(0), R_can, mode='bilinear')[0]  # (C, C, C)

    # Carve canonical wedge exactly after rotation — matches training order (rotate then carve).
    _can_g = canonical_goniometer_axis(rsm_dir, goniometer_axis)
    _can_keep = torch.from_numpy(
        np.fft.fftshift(missing_wedge_mask_3d(
            np.array([0., 1., 0.]), alpha_deg, tuple(vol_can.shape), _can_g,
        ))
    ).to(device)
    Fs = torch.fft.fftshift(torch.fft.fftn(vol_can))
    vol_can = torch.fft.ifftn(torch.fft.ifftshift(Fs * _can_keep.to(Fs.dtype))).real

    arc  = missing_arc_length(rsm_dir, alpha_deg, goniometer_axis)
    cond = sinusoidal_wedge_embedding(arc, dim=_COND_DIM, device=device)   # (1, 1, cond_dim)
    ts   = torch.zeros(1, device=device, dtype=torch.long)

    with torch.no_grad():
        pred_can = _infer_model(
            vol_can[None, None], timestep=ts,
            encoder_hidden_states=cond, return_dict=False,
        )[0].squeeze()   # (C, C, C)

    pred_np = rotate_batch(pred_can.unsqueeze(0), R_inv, mode='bilinear')[0].cpu().numpy()

    if apply_consistency:
        # Enforce Fourier consistency in the natural frame (normalized space):
        # keep measured frequencies from the carved input, fill missing from the model.
        mask = missing_wedge_mask_3d(rsm_dir, alpha_deg, vol_carved_norm_np.shape, goniometer_axis)
        F_pred   = np.fft.fftn(pred_np)
        F_carved = np.fft.fftn(vol_carved_norm_np)
        F_pred[mask] = F_carved[mask]
        pred_np = np.fft.ifftn(F_pred).real

    return pred_np


def _make_inference_viewer(volume, sphere_mask, directions, arcs_deg, alpha_deg, goniometer_axis):
    K_, X_, Y_, Z_ = volume.shape
    mean_vol = volume.mean(axis=0)
    _abs_lo  = float(np.percentile(mean_vol[sphere_mask], 2))
    _abs_hi  = float(np.percentile(mean_vol[sphere_mask], 98))

    def _slice_row(axes, data, x, y, z, kw, row_label):
        axes[0].set_title(f'{row_label}  |  YZ (x={x})', fontsize=8)
        axes[1].set_title(f'{row_label}  |  XZ (y={y})', fontsize=8)
        axes[2].set_title(f'{row_label}  |  XY (z={z})', fontsize=8)

        im = axes[0].imshow(data[x, :, :].T, **kw)
        axes[0].axvline(y, color='tomato',      lw=0.9, alpha=0.85)
        axes[0].axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        axes[0].set_xlabel('Y'); axes[0].set_ylabel('Z')

        axes[1].imshow(data[:, y, :].T, **kw)
        axes[1].axvline(x, color='limegreen',   lw=0.9, alpha=0.85)
        axes[1].axhline(z, color='deepskyblue', lw=0.9, alpha=0.85)
        axes[1].set_xlabel('X'); axes[1].set_ylabel('Z')

        axes[2].imshow(data[:, :, z].T, **kw)
        axes[2].axvline(x, color='limegreen', lw=0.9, alpha=0.85)
        axes[2].axhline(y, color='tomato',    lw=0.9, alpha=0.85)
        axes[2].set_xlabel('X'); axes[2].set_ylabel('Y')
        return im

    def _view(i, j, k_wedge, show_normalized, x, y, z):
        vol_i = volume[i]
        vol_j = volume[j]

        # Per-volume normalization (pipeline convention: stats from raw sphere interior).
        int_i_raw = vol_i[sphere_mask].ravel()
        int_j_raw = vol_j[sphere_mask].ravel()
        mu_i, sigma_i = float(int_i_raw.mean()), float(int_i_raw.std() + 1e-8)
        mu_j, sigma_j = float(int_j_raw.mean()), float(int_j_raw.std() + 1e-8)

        # Normalize then re-zero exterior — matches dataset._load_norm exactly.
        # spherical_crop zeros raw exterior, so raw exterior = 0; after (vol−mu)/sigma
        # the exterior becomes −mu/sigma (large negative shell for positive SAXS data).
        # The model was trained with exterior = 0 in normalized space, so we must re-zero.
        vol_i_norm = (vol_i - mu_i) / sigma_i
        vol_j_norm = (vol_j - mu_j) / sigma_j
        vol_i_norm = vol_i_norm.copy(); vol_i_norm[~sphere_mask] = 0.0
        vol_j_norm = vol_j_norm.copy(); vol_j_norm[~sphere_mask] = 0.0

        # Carve missing wedge of k_wedge from the (now correctly zeroed) normalized volumes.
        carved_i_norm = _apply_wedge(vol_i_norm, directions[k_wedge], alpha_deg, goniometer_axis)
        carved_j_norm = _apply_wedge(vol_j_norm, directions[k_wedge], alpha_deg, goniometer_axis)

        # Model inference (canonical rotation + exact carving + forward pass + Fourier consistency).
        pred_i_norm = _run_model_single(carved_i_norm, directions[k_wedge], alpha_deg, goniometer_axis)
        pred_j_norm = _run_model_single(carved_j_norm, directions[k_wedge], alpha_deg, goniometer_axis)

        if show_normalized:
            disp_carved_i = carved_i_norm;           disp_pred_i = pred_i_norm
            disp_carved_j = carved_j_norm;           disp_pred_j = pred_j_norm
            all_disp = np.concatenate([
                carved_i_norm[sphere_mask], pred_i_norm[sphere_mask],
                carved_j_norm[sphere_mask], pred_j_norm[sphere_mask],
            ])
            vlo = float(np.percentile(all_disp, 1))
            vhi = float(np.percentile(all_disp, 99))
            cbar_label = 'Normalized intensity  (vol − μ_k) / σ_k'
            xlabel     = 'Normalized voxel intensity'
        else:
            disp_carved_i = carved_i_norm * sigma_i + mu_i
            disp_pred_i   = pred_i_norm   * sigma_i + mu_i
            disp_carved_j = carved_j_norm * sigma_j + mu_j
            disp_pred_j   = pred_j_norm   * sigma_j + mu_j
            vlo, vhi  = _abs_lo, _abs_hi
            cbar_label = 'Intensity (arbitrary units)'
            xlabel     = 'Voxel intensity'

        fig = plt.figure(figsize=(13, 22))
        gs  = fig.add_gridspec(5, 3, height_ratios=[1, 1, 1, 1, 0.65], hspace=0.55, wspace=0.3)
        axes_i_in  = [fig.add_subplot(gs[0, c]) for c in range(3)]
        axes_i_out = [fig.add_subplot(gs[1, c]) for c in range(3)]
        axes_j_in  = [fig.add_subplot(gs[2, c]) for c in range(3)]
        axes_j_out = [fig.add_subplot(gs[3, c]) for c in range(3)]
        ax_hist    = fig.add_subplot(gs[4, :])

        kw = dict(cmap='inferno', vmin=vlo, vmax=vhi, aspect='equal', origin='lower')

        im = _slice_row(axes_i_in,  disp_carved_i, x, y, z, kw,
                        f'i={i}  arc={arcs_deg[i]:.1f}°  |  carved input  (k_wedge={k_wedge})')
        _slice_row(axes_i_out, disp_pred_i,   x, y, z, kw,
                   f'i={i}  arc={arcs_deg[i]:.1f}°  |  model output  (k_wedge={k_wedge})')
        _slice_row(axes_j_in,  disp_carved_j, x, y, z, kw,
                   f'j={j}  arc={arcs_deg[j]:.1f}°  |  carved input  (k_wedge={k_wedge})')
        _slice_row(axes_j_out, disp_pred_j,   x, y, z, kw,
                   f'j={j}  arc={arcs_deg[j]:.1f}°  |  model output  (k_wedge={k_wedge})')

        plt.colorbar(im,
                     ax=axes_i_in + axes_i_out + axes_j_in + axes_j_out,
                     shrink=0.25, label=cbar_label)

        # ── Histogram: carved input (dotted) vs model output (solid) ─────────
        ci = disp_carved_i[sphere_mask].ravel();  pi = disp_pred_i[sphere_mask].ravel()
        cj = disp_carved_j[sphere_mask].ravel();  pj = disp_pred_j[sphere_mask].ravel()
        bins = np.linspace(
            np.percentile(np.concatenate([ci, pi, cj, pj]), 0.5),
            np.percentile(np.concatenate([ci, pi, cj, pj]), 99.5), 80,
        )
        ax_hist.hist(ci, bins=bins, histtype='step', color='steelblue', lw=1.5, ls=':',
                     label=f'i={i} carved input')
        ax_hist.hist(pi, bins=bins, histtype='step', color='steelblue', lw=2.0, ls='-',
                     label=f'i={i} model output')
        ax_hist.hist(cj, bins=bins, histtype='step', color='coral',     lw=1.5, ls=':',
                     label=f'j={j} carved input')
        ax_hist.hist(pj, bins=bins, histtype='step', color='coral',     lw=2.0, ls='-',
                     label=f'j={j} model output')
        ax_hist.set_xlabel(xlabel)
        ax_hist.set_ylabel('Count')
        ax_hist.set_title('Carved input (dotted) vs model output (solid) — sphere interior only')
        ax_hist.legend(fontsize=8)

        norm_tag = '  [normalized]' if show_normalized else '  [raw]'
        plt.suptitle(
            f'Model inference{norm_tag}  |  wedge k={k_wedge}  arc={arcs_deg[k_wedge]:.1f}°  '
            f'dir={np.round(directions[k_wedge], 2)}\n'
            f'checkpoint: {CHECKPOINT.parent.name}/{CHECKPOINT.name}',
            fontsize=11,
        )
        plt.show()

    interact(
        _view,
        i=widgets.IntSlider(min=0, max=K_-1, step=1, value=0,
                            description='i (vol A)', continuous_update=False),
        j=widgets.IntSlider(min=0, max=K_-1, step=1, value=K_//2,
                            description='j (vol B)', continuous_update=False),
        k_wedge=widgets.IntSlider(min=0, max=K_-1, step=1, value=K_-1,
                                  description='k_wedge', continuous_update=False),
        show_normalized=widgets.Checkbox(value=True, description='Show normalized'),
        x=widgets.IntSlider(min=0, max=X_-1, step=1, value=X_//2,
                            description='x', continuous_update=False),
        y=widgets.IntSlider(min=0, max=Y_-1, step=1, value=Y_//2,
                            description='y', continuous_update=False),
        z=widgets.IntSlider(min=0, max=Z_-1, step=1, value=Z_//2,
                            description='z', continuous_update=False),
    )


_make_inference_viewer(
    recon_fbp_np, sphere_mask_np, y_directions, arcs_deg, ALPHA_DEG, goniometer_axis
)

Loaded: /myhome/data/smartt/shared/isonet_results/frogbone_corrected/round_10/checkpoint.pt
  alpha=45.0°  conditioning_dim=64  cube_size=64


interactive(children=(IntSlider(value=0, continuous_update=False, description='i (vol A)', max=29), IntSlider(…